# <span style="color:#CC0000">Comprehensive results analysis for rossler system</span>


In [ ]:
# ! Import related libraries
library(ggplot2)
library(plot3D)
library(tidyr)
library(tidyverse)
library(scales)
library(latex2exp)
library(cowplot)
library(ggh4x)
library(ggpubr)
library(gridExtra)
library(patchwork)

library(scales)
library(reticulate)
library(RColorBrewer)
library(stringr)
library(ggplotgui)
library(dplyr)


In [ ]:
# ! import internal functions
setwd(dirname(dirname(dirname(getwd()))))
getwd()
source("./MethodsEvaluation/results_processing_for_parallel_computing_update.R")
ggplot_3d_path <- "./R/ggplot2-3d"
load_r <- list.files(ggplot_3d_path, "*.R")
sapply(load_r, function(i) {
    source(paste0(ggplot_3d_path, "/", i))
})


## <span style = "color:#EF60AD">Common parameters settings for all plotting processes</span>


In [ ]:
# - Specify which dynamical system is under investigation
dynamical_system_name <- "rossler"
true_terms_1 <- list("x2", "x3")
true_terms_2 <- list("x1", "x2")
true_terms_3 <- list("(Intercept)", "x1x3", "x3")


## <span style = "color:#EF60AD">Plotting themes settings for success rate plots N</span>


In [ ]:
ggplot_theme0 <- theme(
    axis.line = element_line(colour = "black"),
    axis.ticks.length = unit(.25, "cm"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    panel.background = element_blank(),
    legend.key = element_blank(),
    legend.text = element_text(size = 16),
    # Changed for legend
    legend.title = element_text(face = "bold", size = 16),
    axis.text = element_text(size = 20),
    axis.title.x = element_text(size = 24),
    axis.title.y = element_text(
        size = 24,
        angle = 90,
        vjust = 0.5
    ),
    plot.title = element_text(size = 24),
    plot.tag = element_text(size = 18),
    legend.position = "none",
    legend.text.align = 0
)
ggplot_theme1 <- ggplot_theme0 + theme(plot.background = element_rect(fill = 0, colour = 0))


In [ ]:
create_separators <- function(x, extra_x, y, extra_y, angle = 45, scale = 1, length = .1) {
    add_y <- length * sin(angle * pi / 180) / 2
    add_x <- length * cos(angle * pi / 180)
    list(
        x = x - add_x * scale, xend = x + add_x * scale + extra_x,
        y = rep(y - add_y * scale - extra_y, length(x)), yend = rep(y + add_y * scale - extra_y / 2, length(x))
    )
}


## <span style="color:#EF60AD">Success rate plot generation (N)</span>


### <span style="color:#FA5524">Generate the comprehensive **_success rate table (N)_** when using **_argos-lasso_**, **_argos-alasso_** and **_bayesian-argos_** algorithms</span>


In [ ]:
# - Parameters settings
root_path <- getwd()
experiment_name_list <- list("argos-lasso", "argos-alasso", "bayesian-argos", "bayesian-alasso-ols", "bayesian-alasso-ridge")
method_list <- list("argos-lasso", "argos-alasso", "bayesian-argos", "bayesian-alasso-ols", "bayesian-alasso-ridge")
function_number <- 3
exp_n <- TRUE
typical_pattern <- "snr49"
num_init <- 100
start <- 2
number_step <- 0.1

# - Generate the total success rate table
total_success_rate_table_n <- generate_total_success_rate_table(
    root_path = root_path,
    experiment_name_list = experiment_name_list,
    method_list = method_list,
    function_number = function_number,
    num_init = num_init,
    dynamical_system_name = dynamical_system_name,
    exp_n = exp_n,
    typical_pattern = typical_pattern,
    true_terms_1 = true_terms_1,
    true_terms_2 = true_terms_2,
    true_terms_3 = true_terms_3,
    start = start,
    number_step = number_step
)

head(total_success_rate_table_n)
tail(total_success_rate_table_n)


### <span style="color:#FA5524">Generate the **_success rate plot (N)_** when using **_argos-lasso_**, **_argos-alasso_** and **_bayesian-argos_** algorithms</span>


In [ ]:
# - Parameters settings for success rate plot (N)
total_correct <- total_success_rate_table_n
models_name <- unique(total_correct$Model)
new_levels <- models_name[c(1, 2, 3, 4, 5)]
total_correct$Model <- factor(total_correct$Model, levels = new_levels)
n_seq <- seq(2, 5, by = 0.1)
rect1 <- data.frame(xmin = -Inf, xmax = Inf, ymin = 0.8, ymax = Inf)
colors_correct <- c("#CC0000", "#054D91", "#F0B823", "#FA5524", "#EF60AD")
shapes <- c(15, 16, 17, 18, 19)

x_labels <- c(
    expression(paste("10"^"2")),
    expression(paste("10"^"2.5")),
    expression(paste("10"^"3")),
    expression(paste("10"^"3.5")),
    expression(paste("10"^"4")),
    expression(paste("10"^"4.5")),
    expression(paste("10"^"5"))
)

x_breaks <- pretty_breaks()(n_seq)
y_breaks <-
    y_labels <- pretty_breaks()(c(0, max(total_correct$Value)))

prob_increase_n <-
    ggplot() +
    geom_hline(yintercept = 0.8, lty = 2) +
    geom_rect(data = rect1, aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax), alpha = 0.2, fill = "#9de0e6") +
    geom_point(
        data = total_correct,
        aes(
            x = eta,
            y = Value,
            fill = Model,
            col = Model,
            shape = Model
        ), size = 4
    ) +
    labs(
        y = "Success Rate",
        x = expression(italic("n"))
    ) +
    scale_x_continuous(
        labels = x_labels,
        breaks = x_breaks
    ) +
    scale_y_continuous(
        labels = y_labels,
        breaks = y_breaks,
        limits = c(0, max(y_labels))
    ) +
    ggplot_theme1 +
    labs(title = paste(dynamical_system_name), tag = "a") +
    theme(legend.position = "bottom") +
    scale_fill_manual(values = colors_correct, breaks = models_name, labels = models_name) +
    scale_colour_manual(values = colors_correct, breaks = models_name, labels = models_name) +
    scale_shape_manual(values = shapes, breaks = models_name, labels = models_name)



system_n <- arrangeGrob(prob_increase_n)
grid.arrange(system_n, ncol = 1)


## <span style="color:#EF60AD">Success rate plot generation (SNR)</span>


### <span style="color:#FA5524">Generate the comprehensive **_success rate table (SNR)_** when using **_argos-lasso_**, **_argos-alasso_** and **_bayesian-argos_** algorithms</span>


In [ ]:
# - Parameters settings
root_path <- getwd()
experiment_name_list <- list("argos-lasso", "argos-alasso", "bayesian-argos", "bayesian-alasso-ols", "bayesian-alasso-ridge")
method_list <- list("argos-lasso", "argos-alasso", "bayesian-argos", "bayesian-alasso-ols", "bayesian-alasso-ridge")
function_number <- 3
exp_n <- FALSE
typical_pattern <- "n5000"
num_init <- 100
start <- 1
number_step <- 1

# - Generate the total success rate table
total_success_rate_table_snr <- generate_total_success_rate_table(
    root_path = root_path,
    experiment_name_list = experiment_name_list,
    method_list = method_list,
    function_number = function_number,
    num_init = num_init,
    dynamical_system_name = dynamical_system_name,
    exp_n = exp_n,
    typical_pattern = typical_pattern,
    true_terms_1 = true_terms_1,
    true_terms_2 = true_terms_2,
    true_terms_3 = true_terms_3,
    start = start,
    number_step = number_step
)

head(total_success_rate_table_snr)
tail(total_success_rate_table_snr)


### <span style="color:#FA5524">Generate the **_success rate plot (N)_** when using **_argos-lasso_**, **_argos-alasso_** and **_bayesian-argos_** algorithms</span>


In [ ]:
colors_correct <- c("#CC0000", "#054D91", "#F0B823", "#FA5524", "#EF60AD")
shapes <- c(15, 16, 17, 18, 19)
rect1 <- data.frame(xmin = -Inf, xmax = Inf, ymin = 0.8, ymax = Inf)

total_correct <- total_success_rate_table_snr
for (i in seq(62, nrow(total_correct), 62)) {
    total_correct$snr[i] <- 73
}
models_name <- unique(total_correct$Model)
new_levels <- models_name[c(1, 2, 3, 4, 5)]
total_correct$Model <- factor(total_correct$Model, levels = new_levels)
# total_correct <- arrange(total_correct, Model)
# models_name <- new_levels[c(1, 2)]

x_labels <- x_breaks <- seq(1, 73, by = 12)
x_labels[length(x_labels)] <- TeX("$\\infty$")

y_labels <-
    y_breaks <-
    pretty_breaks()(c(min(total_correct$Value), max(total_correct$Value)))

xstart <- 65.5
xend <- 69.5
extra_x <- 1
y_sep <- min(total_correct$Value) - 0.05 * (min(total_correct$Value))
myseg <- create_separators(c(xstart, xend), extra_x = 1, y = y_sep, extra_y = 0.1, angle = 75)

prob_increase_snr <-
    ggplot() +
    geom_hline(yintercept = 0.8, lty = 2) +
    geom_rect(data = rect1, aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax), alpha = 0.2, fill = "#9de0e6") +
    geom_point(
        data = total_correct,
        aes(
            x = snr,
            y = Value,
            fill = Model,
            col = Model,
            shape = Model
        ), size = 4
    ) +
    labs(
        y = "Success Rate",
        x = TeX("SNR(dB)")
    ) +
    scale_x_continuous(
        limits = c(
            min(x_breaks),
            max(x_breaks)
        ),
        labels = x_labels,
        breaks = x_breaks
    ) +
    scale_y_continuous(
        labels = y_labels,
        breaks = y_breaks,
        limits = c(NA, max(y_breaks))
    ) +
    ggplot_theme1 +
    labs(title = paste(dynamical_system_name), tag = "b") +
    theme(legend.position = "bottom") +
    scale_fill_manual(values = colors_correct, breaks = models_name, labels = models_name) +
    scale_colour_manual(values = colors_correct, breaks = models_name, labels = models_name) +
    scale_shape_manual(values = shapes, breaks = models_name, labels = models_name) +
    guides(x = guide_axis_truncated(
        trunc_lower = c(-Inf, xend + extra_x / 2),
        trunc_upper = c(xstart + extra_x / 2, Inf)
    )) +
    annotate("segment",
        x = myseg$x, xend = myseg$xend,
        y = myseg$y + 0.05, yend = myseg$yend
    ) +
    coord_cartesian(clip = "off", ylim = c(-0.0005, NA))

prob_increase_snr + labs(title = paste(dynamical_system_name), tag = "b") + theme(legend.position = "bottom")


## <span style="color:#EF60AD">Running time comparison between different system identification algorithms</span>


### <span style="color:#FA5524">Generate the **_total results summary dataframe (N)_** when using **_argos-lasso_**, **_argos-alasso_** and **_bayesian-argos_** algorithms</span>


In [ ]:
# - Parameters settings
root_path <- getwd()
experiment_name_list <- list("argos-lasso", "argos-alasso", "bayesian-argos", "bayesian-alasso-ols", "bayesian-alasso-ridge")
method_list <- list("argos-lasso", "argos-alasso", "bayesian-argos", "bayesian-alasso-ols", "bayesian-alasso-ridge")
function_number <- 3
exp_n <- TRUE
typical_pattern <- "snr49"
num_init <- 100


In [ ]:
# - Generate the total results summary dataframe
total_results_summary_df <- enhance_total_results_summary_df(
    root_path = root_path,
    experiment_name_list = experiment_name_list,
    method_list = method_list,
    function_number = function_number,
    num_init = num_init,
    dynamical_system_name = dynamical_system_name,
    exp_n = exp_n,
    typical_pattern = typical_pattern
)

# - Show the processed total dataset
head(total_results_summary_df)


### <span style="color:#FA5524">Synthesis information from the **_total results summary dataframe (N)_** to calculate the time complexity of different methods</span>


In [ ]:
methods_running_time_table <- create_time_complexity_table(
    total_results_summary_df = total_results_summary_df,
    type_1_method_list = c("alasso", "lasso"),
    type_1_method_cpu_number = 20,
    type_2_method_list = c("bayesian-argos", "bayesian-alasso-ols", "bayesian-alasso-ridge"),
    type_2_method_cpu_number = 4
)
head(methods_running_time_table)
tail(methods_running_time_table)
dim(methods_running_time_table)


In [ ]:
selected_eta_list <- seq(3, 5, by = 0.5)
methods_running_time_table <- methods_running_time_table %>%
  dplyr::filter(eta %in% selected_eta_list)

methods_running_time_table_lasso <- methods_running_time_table %>%
  dplyr::filter(method == "lasso")
methods_running_time_table_lasso$eta <- as.numeric(methods_running_time_table_lasso$eta) * 2 - 5

methods_running_time_table_alasso <- methods_running_time_table %>%
  dplyr::filter(method == "alasso")
methods_running_time_table_alasso$eta <- as.numeric(methods_running_time_table_alasso$eta) * 2 - 5

methods_running_time_table_bayesian_argos <- methods_running_time_table %>%
  dplyr::filter(method == "bayesian-argos")
methods_running_time_table_bayesian_argos$eta <- as.numeric(methods_running_time_table_bayesian_argos$eta) * 2 - 5

methods_running_time_table_bayesian_argos <- methods_running_time_table %>%
  dplyr::filter(method == "bayesian-argos")
methods_running_time_table_bayesian_argos$eta <- as.numeric(methods_running_time_table_bayesian_argos$eta) * 2 - 5

methods_running_time_table_bayesian_alasso_ols <- methods_running_time_table %>%
  dplyr::filter(method == "bayesian-alasso-ols")
methods_running_time_table_bayesian_alasso_ols$eta <- as.numeric(methods_running_time_table_bayesian_alasso_ols$eta) * 2 - 5

methods_running_time_table_bayesian_alasso_ridge <- methods_running_time_table %>%
  dplyr::filter(method == "bayesian-alasso-ridge")
methods_running_time_table_bayesian_alasso_ridge$eta <- as.numeric(methods_running_time_table_bayesian_alasso_ridge$eta) * 2 - 5


### <span style="color:#F0B823">Plot parameters settings for the time complexity of different methods</span>


In [ ]:
colors <-
    c(
        "#CC0000",
        "#054D91",
        "#F0B823",
        "#FA5524",
        "#EF60AD" # F0B823
    )

x_labels <- c(
    expression(paste("10"^"3")),
    expression(paste("10"^"3.5")),
    expression(paste("10"^"4")),
    expression(paste("10"^"4.5")),
    expression(paste("10"^"5"))
)

y_breaks <- trans_breaks("log10", function(x) {
    10^x
})(c(min(methods_running_time_table$run_time), max(methods_running_time_table$run_time)))
y_breaks <-
    y_labels <- pretty_breaks()(c(0, max(methods_running_time_table$run_time)))

breaks_log10 <- function(x) {
    low <- floor(log10(min(x)))
    high <- ceiling(log10(max(x)))

    10^(seq.int(low, high))
}


In [ ]:
runtime_ggplot <-
    ggplot() +
    geom_boxplot(data = methods_running_time_table, aes(x = eta, y = run_time, fill = method), lwd = 0.3) +
    geom_smooth(
        data = methods_running_time_table_alasso, aes(x = eta - 0.2, y = run_time),
        method = "lm", formula = y ~ stats::poly(x, 2, raw = TRUE), alpha = 0.2, lty = 2, color = colors[2], se = FALSE
    ) +
    geom_smooth(
        data = methods_running_time_table_lasso, aes(x = eta + 0.2, y = run_time),
        method = "lm", formula = y ~ stats::poly(x, 2, raw = TRUE), alpha = 0.2, lty = 2, color = colors[1], se = FALSE
    ) +
    geom_smooth(
        data = methods_running_time_table_bayesian_argos, aes(x = eta, y = run_time),
        method = "lm", formula = y ~ stats::poly(x, 2, raw = TRUE), alpha = 0.2, lty = 2, color = colors[3], se = FALSE
    ) +
    geom_smooth(
        data = methods_running_time_table_bayesian_alasso_ols, aes(x = eta, y = run_time),
        method = "lm", formula = y ~ stats::poly(x, 2, raw = TRUE), alpha = 0.2, lty = 2, color = colors[4], se = FALSE
    ) +
    geom_smooth(
        data = methods_running_time_table_bayesian_alasso_ridge, aes(x = eta, y = run_time),
        method = "lm", formula = y ~ stats::poly(x, 2, raw = TRUE), alpha = 0.2, lty = 2, color = colors[5], se = FALSE
    ) +
    # annotate(
    #     geom = "text", x = 1, y = 3200,
    #     label = TeX("$\\sim 0.42(log_{10}n)^2$", output = "character"),
    #     parse = TRUE, size = 9
    # ) +
    # annotate(
    #     geom = "text", x = 1, y = 650,
    #     label = TeX("$\\sim 0.21(log_{10}n)^2$", output = "character"),
    #     parse = TRUE, size = 9
    # ) +
    # annotate(
    #     geom = "text", x = 2.5, y = 250,
    #     label = TeX("$\\sim 0.724(log10n)^2$", output = "character"),
    #     parse = TRUE, size = 9
    # ) +
    scale_y_log10(
        breaks = breaks_log10,
        labels = trans_format(log10, math_format(10^.x))
    ) +
    scale_x_discrete(
        drop = FALSE,
        labels = x_labels
    ) +
    labs(
        x = TeX("$\\textit{n}$"),
        y = "Time [s]",
        tag = "c",
        title = dynamical_system_name
    ) +
    theme(
        axis.line = element_line(colour = "black"),
        axis.ticks.length = unit(.25, "cm"),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        panel.border = element_blank(),
        panel.background = element_blank(),
        legend.key = element_blank(),
        legend.text = element_text(size = 16),
        # Changed for legend
        legend.title = element_text(face = "bold", size = 16),
        axis.text = element_text(size = 20),
        axis.title.x = element_text(size = 28),
        axis.title.y = element_text(
            size = 28,
            angle = 90,
            vjust = 0.5
        ),
        plot.title = element_text(size = 24), # vjust = -5
        plot.tag = element_text(size = 20, face = "bold"), # vjust = -10,
        legend.position = "none",
        # legend.text.align = 0,
        # plot.margin = unit(c(-35, 0, 0, 0), "pt")
    ) +
    theme(legend.position = "bottom") +
    scale_fill_manual(values = colors[c(2, 3, 1, 4, 5)])
# scale_colour_manual(values = colors)

runtime_ggplot


## <span style="color:#EF60AD">Number of identified terms (N) plot generation</span>


### <span style="color:#FA5524">Generate the **_total results summary dataframe (N)_** when using **_argos-lasso_**, **_argos-alasso_** and **_bayesian-argos_** algorithms</span>


In [ ]:
# - Parameters settings
root_path <- getwd()
experiment_name_list <- list("argos-lasso", "argos-alasso", "bayesian-argos", "bayesian-alasso-ols", "bayesian-alasso-ridge")
method_list <- list("argos-lasso", "argos-alasso", "bayesian-argos", "bayesian-alasso-ols", "bayesian-alasso-ridge")
function_number <- 3
exp_n <- TRUE
typical_pattern <- "snr49"
num_init <- 100


In [ ]:
# - Summary dataframe generation
total_results_summary_df <- enhance_total_results_summary_df(
    root_path = root_path,
    experiment_name_list = experiment_name_list,
    method_list = method_list,
    function_number = function_number,
    num_init = num_init,
    dynamical_system_name = dynamical_system_name,
    exp_n = exp_n,
    typical_pattern = typical_pattern
)

# - Show the processed total dataset
head(total_results_summary_df)
tail(total_results_summary_df)


### <span style="color:#FA5524">Generate the **_number of identified terms plot (N)_** when using **_argos-lasso_** algorithm</span>


In [ ]:
# - Select the core outputs to form a dataset for plotting purposes
selected_eta_list <- seq(2, 5, by = 0.5)
selected_results_summary_df <- total_results_summary_df %>%
    dplyr::filter(method == "lasso") %>%
    dplyr::filter(eta %in% selected_eta_list)

head(selected_results_summary_df)
dim(selected_results_summary_df)

terms_min <- min(selected_results_summary_df$num_of_identified_terms)
terms_max <- max(selected_results_summary_df$num_of_identified_terms)


In [ ]:
# - ggplot parameters settings for plotting identified terms numbers when n increase
colors_plot <- c(
  "#cbc9e2",
  "#9e9ac8",
  "#766bb1"
)
x_axis_breaks_n <- c("2", "2.5", "3", "3.5", "4", "4.5", "5")
x_axis_labels_n <- c(
  expression(10^2), expression(10^{
    2.5
  }), expression(10^3),
  expression(10^{
    3.5
  }), expression(10^4), expression(10^{
    4.5
  }),
  expression(10^5)
)

x_axis_breaks_snr <- c(1, 13, 25, 37, 49, 61, 73)
x_axis_labels_snr <- c(1, 13, 25, 37, 49, 61, expression(infinity))

plot_title <- 25

equation_labels <- my_expressions <- c(
  expression(dot(x)[1]),
  expression(dot(x)[2]),
  expression(dot(x)[3])
)
x_axis_text_size <- 20
y_axis_text_size <- 20
y_axis_label_size <- 24
x_axis_label_size <- 24
# a simple function to help make the segments
create_separators <- function(x, extra_x, y, extra_y, angle = 45, scale = 1, length = .1) {
  add_y <- length * sin(angle * pi / 180) / 2
  add_x <- length * cos(angle * pi / 180)
  list(
    x = x - add_x * scale, xend = x + add_x * scale + extra_x,
    y = rep(y - add_y * scale - extra_y, length(x)), yend = rep(y + add_y * scale - extra_y / 2, length(x))
  )
}


In [ ]:
num_of_terms_n_lasso <-
    ggplot(selected_results_summary_df, aes(x = eta, y = num_of_identified_terms, fill = function_indicator)) +
    geom_boxplot() +
    ggtitle("argos-lasso") +
    xlab("n") +
    ylab("Number of terms") +
    scale_x_discrete(labels = x_axis_labels_n) +
    scale_y_continuous(limits = c(terms_min, terms_max)) +
    coord_cartesian(ylim = c(terms_min, terms_max)) +
    scale_color_manual(
        name = "Equation",
        breaks = c("xdot", "ydot", "zdot"),
        labels = equation_labels,
        values = colors_plot
    ) +
    scale_fill_manual(
        name = "Equation",
        breaks = c("xdot", "ydot", "zdot"),
        labels = equation_labels,
        values = colors_plot
    ) +
    theme(
        plot.title = element_text(size = plot_title),
        axis.text.x = element_text(size = x_axis_text_size),
        axis.text.y = element_text(size = y_axis_text_size),
        axis.title.y = element_text(
            size = y_axis_label_size,
            angle = 90,
            vjust = 0.5
        ),
        axis.title.x = element_text(size = x_axis_label_size, face = "italic"),
        panel.background = element_blank(),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        legend.position = "none",
        axis.line.x = element_line(color = "black"),
        axis.line.y = element_line(color = "black")
    )


In [ ]:
num_of_terms_n_lasso


## <span style="color:#EF60AD">Organise the related plots into a comprehensive results analysis plot</span>


### <span style="color:#FA5524">Generate the **_success rate plot (N)_** when using **_argos-lasso_**, **_argos-alasso_** and **_bayesian-argos_** algorithms</span>


In [ ]:
gglegend <- get_legend(prob_increase_snr)

# grid.arrange(system_n, ncol = 1)
system_comprehensive <-
    arrangeGrob(prob_increase_n + theme(legend.position = "none"),
        prob_increase_snr + theme(legend.position = "none"),
        runtime_ggplot + theme(legend.position = "none"),
        nrow = 1
    )
# system_comprehensive <- grid.arrange(system_comprehensive, gglegend, ncol = 1, heights = c(15, 1))
grid.arrange(system_comprehensive)
# ggsave(plot = system_comprehensive, filename = "/Users/yuzhengzhang/Desktop/system_comprehensive.png", width = 12, height = 6)
ggsave(plot = system_comprehensive, filename = "/Users/yuzhengzhang/Desktop/New_results/rossler_results.png", width = 20, height = 6)


In [ ]:
gglegend <- get_legend(prob_increase_snr)

# grid.arrange(system_n, ncol = 1)
system_comprehensive <-
    arrangeGrob(prob_increase_n + theme(legend.position = "none"),
        prob_increase_snr + theme(legend.position = "none"),
        runtime_ggplot + theme(legend.position = "none"),
        nrow = 1
    )
system_comprehensive <- grid.arrange(system_comprehensive, gglegend, ncol = 1, heights = c(15, 1))
grid.arrange(system_comprehensive)
# ggsave(plot = system_comprehensive, filename = "/Users/yuzhengzhang/Desktop/system_comprehensive.png", width = 12, height = 6)
ggsave(plot = system_comprehensive, filename = "/Users/yuzhengzhang/Desktop/Manuscripts/MS-Annual-Review-2024/presentation/img/identification-results-slide/lorenz_results.png", width = 20, height = 6)
